# Replicate scatter of EPMA reference-material analyses versus the reported error

**Question:** In the 85 replicate EPMA analyses of reference material 7802 Apatite, does the reported `errpct` match the observed replicate scatter for each element?

**Data:** `data/EMPA_Apatite_Standards.csv` from the USGS data release "Apatite and monazite geochemistry data at Mountain Pass, California, 2024" (Benson and Watts), https://doi.org/10.5066/P1LGUGBG, accessed 2026-11-03, CC0.

**Data and code availability:** the data are committed in `data/` exactly as downloaded, and the original release is at the DOI above. This notebook reads only `data/EMPA_Apatite_Standards.csv`. It runs from top to bottom in the course environment rebuilt from `pyproject.toml` and `uv.lock`, and it downloads nothing.

## Parameters

**Purpose:** keep every label and every number that encodes a choice in one place, so that later cells use the same values.

**Scientific choice:** `MIN_DETECTION_FRACTION` is 1.0, so an element is used only if it is detected in all of the replicate analyses. The data blank a value below the detection limit, so a partly detected element would leave only its detected values, which truncates the scatter and biases the mean upward. The labels are constants so that a change in the file's wording is fixed in one line.

**Expected output:** none; the cell only defines constants.

In [1]:
DATA_PATH = "data/EMPA_Apatite_Standards.csv"  # path relative to the notebook's folder
STANDARD_LABEL = "7802 Apatite"  # value of `Sample` for the replicate analyses of the reference material
DETECTION_LIMIT_LABEL = "Detection Limit"  # value of `Sample` for the row that holds the detection limits
MIN_DETECTION_FRACTION = 1.0  # fraction of the replicate analyses (no unit) in which an element must be detected; blanks below the detection limit truncate the scatter
NOT_MEASURED_COLUMNS = ["O", "Total"]  # `O` is calculated and `Total` is a sum (data dictionary), so neither is a measured element

## Load the data

**Purpose:** load the file from its relative path and separate the replicate analyses from the detection-limit row.

**What one row is:** one replicate EPMA analysis of the reference material `7802 Apatite`, except one row that holds the detection limit of each element. Units: `*_wtpct` is weight percent, `*_errpct` is percent (the data dictionary says only "Error of ... analysis, percent"), and `*_TDI` is percent (time-dependent intensity correction).

**Scientific choice:** the `Detection Limit` row is not a measurement, so it is kept apart and never enters the replicate scatter. Any row with another label would be reported, not silently dropped.

**Expected output:** 86 rows in the file, of which 85 are replicate analyses and 1 is the detection-limit row, and 0 rows with any other label.

In [2]:
import pandas as pd

data = pd.read_csv(DATA_PATH)
is_detection_limit = data["Sample"] == DETECTION_LIMIT_LABEL
is_replicate = data["Sample"] == STANDARD_LABEL

detection_limits = data[is_detection_limit].iloc[0]  # wt% for *_wtpct columns, one value per column
replicates = data[is_replicate].reset_index(drop=True)  # one row per replicate analysis

print(f"{len(data)} rows in the file: {len(replicates)} replicate analyses of {STANDARD_LABEL}, {is_detection_limit.sum()} detection-limit row")
print(f"rows with any other label: {(~is_detection_limit & ~is_replicate).sum()}")

86 rows in the file: 85 replicate analyses of 7802 Apatite, 1 detection-limit row
rows with any other label: 0


## Detections per element

**Purpose:** count, for each element, how many of the 85 replicate analyses have a reported `*_wtpct` value, and let code pick the elements that pass `MIN_DETECTION_FRACTION`.

**Scientific choice:** a blank `*_wtpct` means the analysis was below the detection limit, so "detected" means "not blank". `O` and `Total` are not measured elements and are left out of the candidates. The same table also counts the `*_errpct` values that are reported and the ones that are negative, because a negative error percent cannot be an error and would break any comparison with scatter.

**Expected output:** a table with one row per candidate element (Ce, Mn, Ca, Sr, Ba, Na, P, S, Cl, F), then the elements used and the elements excluded. The elements Ce, Ca, Sr, Na, P, S and F should be detected 85 of 85 times and Mn, Ba and Cl fewer times; negative `errpct` values should appear only in elements that are excluded.

In [3]:
candidate_elements = [
    col.removesuffix("_wtpct")
    for col in replicates.columns
    if col.endswith("_wtpct") and col.removesuffix("_wtpct") not in NOT_MEASURED_COLUMNS
]

detection_table = pd.DataFrame(
    {
        "n_detected": [replicates[f"{el}_wtpct"].notna().sum() for el in candidate_elements],  # count of replicates with a wt% value
        "detection_limit_wtpct": [detection_limits[f"{el}_wtpct"] for el in candidate_elements],  # wt%
        "n_errpct_reported": [replicates[f"{el}_errpct"].notna().sum() for el in candidate_elements],  # count
        "n_errpct_negative": [(replicates[f"{el}_errpct"] < 0).sum() for el in candidate_elements],  # count
    },
    index=candidate_elements,
)
detection_table["detection_fraction"] = detection_table["n_detected"] / len(replicates)  # no unit
detection_table["used"] = detection_table["detection_fraction"] >= MIN_DETECTION_FRACTION

elements_used = detection_table.index[detection_table["used"]].tolist()
elements_excluded = detection_table.index[~detection_table["used"]].tolist()

print(detection_table.to_string())
print(f"elements used: {elements_used}")
print(f"elements excluded: {elements_excluded}")

    n_detected  detection_limit_wtpct  n_errpct_reported  n_errpct_negative  detection_fraction   used
Ce          85                  0.077                 85                  0            1.000000   True
Mn          32                  0.025                 85                  6            0.376471  False
Ca          85                  0.061                 85                  0            1.000000   True
Sr          85                  0.061                 85                  0            1.000000   True
Ba           2                  0.317                 85                 38            0.023529  False
Na          85                  0.012                 85                  0            1.000000   True
P           85                  0.021                 85                  0            1.000000   True
S           85                  0.009                 85                  0            1.000000   True
Cl           5                  0.009                 85                 

**Problems found on loading, and what was done about each:**

- **Blank cells.** A blank `*_wtpct` means the value was below the detection limit, so blanks were left as missing values and not replaced by zero or by the detection limit, which would invent a number that was not measured.
- **Detections per element.** Ce, Ca, Sr, Na, P, S and F were detected in all 85 analyses and are used; Mn (32 of 85), Cl (5 of 85) and Ba (2 of 85) were detected too rarely, because the replicates that remain would be only the high values and would understate the scatter, so they are excluded by `MIN_DETECTION_FRACTION`.
- **Negative `errpct`.** Negative values appear only in the excluded elements (Mn 6, Cl 30, Ba 38), and none appear in the seven used elements, so nothing was removed or corrected; the excluded elements also report an `errpct` for all 85 analyses even where no value was detected, which shows that their `errpct` is not a meaningful error.
- **The `Detection Limit` row and the non-measured columns.** The detection-limit row was kept apart from the replicates, and `O` (calculated) and `Total` (a sum) were left out of the candidate elements.